# Parquet Benchmarks – Backtesting Experiments

## Libraries

In [1]:
from fabric import ThreadingGroup, Connection
from chi import server, ssh
import os
import warnings
import time
import sys
sys.path.append('../scripts')
from utilities import *
warnings.filterwarnings('ignore')

## Configurations

In [2]:
sshUser = "ubuntu"
client_ip = "64.131.114.143"

storageType = {"HDD": "/HDD", "NVME": "/NVME_RAID-0"}
WORKING_DIR = storageType["NVME"]

targetClients = [1, 2, 4, 8, 16, 32, 64]

## Initialize environment

In [3]:
# These groups are for processing in parallel 
clientConnection = makeConnection(client_ip, user=sshUser)

# Update & Upgrade latest packages
clientConnection.run("sudo apt update && sudo apt upgrade -y", hide=True)
clientConnection.run("sudo reboot", disown=True)
clientConnection.close()

Wait for all server to come online after reboot

In [4]:
# Buffer of reboot (no more +1 min shutdown-schedule delay)
time.sleep(240)
server.wait_for_tcp(client_ip, 22, timeout=1200)

## Benchmark server

Prepare benchmark server

In [5]:
clientConnection.run(f"sudo chmod -R 777 {WORKING_DIR}", hide=True)
clientConnection.close()

## Benchmark Client (Backtester)

In [6]:
clientConnection.put("../scripts/backtesterSkipping.py")
clientConnection.put("../scripts/initBacktestClient.sh")
clientConnection.run("bash initBacktestClient.sh", hide=True)
clientConnection.close()

## Bootstrap Database

In [7]:
clientConnection.put("../scripts/restoreDB.sh")
clientConnection.close()

# Restore DB
clientConnection.run(f"bash restoreDB.sh MYSTIC@IIT CSV BACKTESTER {WORKING_DIR}", hide=True)
clientConnection.close()

# Keep checking if job is done running in tmux window
checkTmuxJobIsDone(clientConnection)

## Experiments

**Perform experiments**

In [8]:
runWorkload("BACKTEST", targetClients, "127.0.0.1", 0000, "CSV", [client_ip], 100, 10, clientConnection, WORKING_DIR, "ubuntu", 1000000, 900, f"{WORKING_DIR}/BacktestData.csv", sleepInterval=60)

[Benchmark Config]
  Target Clients     : 1
  Total Machines     : 1
  Machines Used      : 1
  Threads per Machine: 1
  Total Clients Actual : 1
[Benchmark: 1] - Schedule workload on 1776437112
[Benchmark: 1] [Client: 64.131.114.143] - Time now: 1776437002 | Delta: 110
[Benchmark: 1] [Client: 64.131.114.143] - Downloaded 1 result files
[Benchmark Config]
  Target Clients     : 2
  Total Machines     : 1
  Machines Used      : 1
  Threads per Machine: 2
  Total Clients Actual : 2
[Benchmark: 2] - Schedule workload on 1776438259
[Benchmark: 2] [Client: 64.131.114.143] - Time now: 1776438148 | Delta: 111
[Benchmark: 2] [Client: 64.131.114.143] - Downloaded 1 result files
[Benchmark Config]
  Target Clients     : 4
  Total Machines     : 1
  Machines Used      : 1
  Threads per Machine: 4
  Total Clients Actual : 4
[Benchmark: 4] - Schedule workload on 1776439406
[Benchmark: 4] [Client: 64.131.114.143] - Time now: 1776439295 | Delta: 111
[Benchmark: 4] [Client: 64.131.114.143] - Downloade